# Pathshongi BanglaBERT — chapter-wise training

This notebook trains one BanglaBERT sequence classifier with 13 Physics chapter labels. Its inputs are only Bengali questions and chapter IDs. It never loads answers, answer spans, explanations, or answer passages. The exported model predicts a chapter; it does not generate an answer.

In [ ]:
import sys, torch
assert 'google.colab' in sys.modules, 'Run this notebook in Google Colab'
assert torch.cuda.is_available(), 'In Colab choose Runtime > Change runtime type > T4 GPU'
!pip -q install 'transformers==4.46.3' 'datasets==3.2.0' 'accelerate==1.2.1' 'sentencepiece>=0.2.0'

## 1. Upload the question-only dataset
Upload `dataset/physics_questions_234.json` from the repository.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'physics_questions_234.json' in uploaded, 'Upload physics_questions_234.json'

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib, json
dataset_path = Path('physics_questions_234.json')
dataset_bytes = dataset_path.read_bytes()
document = json.loads(dataset_bytes.decode('utf-8'))
rows = document['questions']
forbidden = {'answer', 'answers', 'answer_start', 'direct_answer', 'reference_answer', 'display_answer', 'explanation', 'context'}
assert document['task'] == 'physics-chapter-classification'
assert len(rows) == 234
assert len({row['question'] for row in rows}) == 234
assert Counter(int(row['chapter_id']) for row in rows) == {chapter: 18 for chapter in range(1, 14)}
for row in rows:
    assert not forbidden.intersection(row), f"Answer-bearing field in {row['id']}"
    assert int(row['label']) == int(row['chapter_id']) - 1
dataset_sha256 = hashlib.sha256(dataset_bytes).hexdigest()
print('Validated question-only data:', len(rows), 'rows')
print('SHA-256:', dataset_sha256)

## 2. Make a reproducible chapter-balanced split
Each chapter contributes 12 training, 3 validation, and 3 final test questions.

In [ ]:
import random
from datasets import Dataset, DatasetDict
by_chapter = defaultdict(list)
for row in rows:
    by_chapter[int(row['chapter_id'])].append(row)
train_rows, validation_rows, test_rows = [], [], []
for chapter_id in range(1, 14):
    chapter_rows = sorted(by_chapter[chapter_id], key=lambda row: row['id'])
    random.Random(4200 + chapter_id).shuffle(chapter_rows)
    train_rows.extend(chapter_rows[:12])
    validation_rows.extend(chapter_rows[12:15])
    test_rows.extend(chapter_rows[15:])
def model_row(row):
    return {'text': row['question'], 'label': int(row['chapter_id']) - 1, 'example_id': row['id']}
raw = DatasetDict({
    'train': Dataset.from_list([model_row(row) for row in train_rows]),
    'validation': Dataset.from_list([model_row(row) for row in validation_rows]),
    'test': Dataset.from_list([model_row(row) for row in test_rows]),
})
assert len(raw['train']) == 156 and len(raw['validation']) == 39 and len(raw['test']) == 39
raw

## 3. Tokenize and train BanglaBERT

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
BASE_MODEL = 'csebuetnlp/banglabert'
MAX_LENGTH = 128
chapter_titles = {int(row['chapter_id']): row['chapter_title'] for row in rows}
id2label = {index: f"chapter_{index + 1:02d}" for index in range(13)}
label2id = {label: index for index, label in id2label.items()}
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=13, id2label=id2label, label2id=label2id
)
tokenized = raw.map(
    lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH),
    batched=True, remove_columns=['text', 'example_id'],
)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
set_seed(42)
def compute_metrics(result):
    predicted = np.argmax(result.predictions, axis=-1)
    truth = result.label_ids
    f1_values = []
    for label in range(13):
        tp = int(((predicted == label) & (truth == label)).sum())
        fp = int(((predicted == label) & (truth != label)).sum())
        fn = int(((predicted != label) & (truth == label)).sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1_values.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return {'accuracy': float((predicted == truth).mean()), 'macro_f1': float(np.mean(f1_values))}
WORK_DIR = '/content/pathshongi-banglabert-physics-chapters'
training_args = TrainingArguments(
    output_dir=WORK_DIR, learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=12, eval_strategy='epoch', save_strategy='epoch',
    logging_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='macro_f1', greater_is_better=True, save_total_limit=2,
    fp16=True, report_to='none', seed=42, data_seed=42,
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'], tokenizer=tokenizer,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()

## 4. Evaluate once on the untouched test split

In [ ]:
test_metrics = trainer.evaluate(tokenized['test'], metric_key_prefix='test')
print(json.dumps(test_metrics, indent=2))
predicted = np.argmax(trainer.predict(tokenized['test']).predictions, axis=-1)
truth = np.asarray(raw['test']['label'])
confusion = np.zeros((13, 13), dtype=int)
for actual, guess in zip(truth, predicted):
    confusion[actual, guess] += 1
print('Confusion matrix (rows=true, columns=predicted):')
print(confusion)

## 5. Export, zip, and download the checkpoint

In [ ]:
from datetime import datetime, timezone
import shutil
EXPORT_DIR = Path('/content/pathshongi-banglabert-physics-chapters-export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)
metadata = {
    'task': 'physics-chapter-classification',
    'base_model': BASE_MODEL,
    'contains_answers': False,
    'training_inputs': ['question', 'chapter_id'],
    'chapters': {str(key): value for key, value in chapter_titles.items()},
    'split_sizes': {'train': 156, 'validation': 39, 'test': 39},
    'dataset_sha256': dataset_sha256,
    'max_length': MAX_LENGTH,
    'test_metrics': test_metrics,
    'created_at': datetime.now(timezone.utc).isoformat(),
}
(EXPORT_DIR / 'training_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)
archive = shutil.make_archive('/content/pathshongi-banglabert-physics-chapters', 'zip', EXPORT_DIR)
print('Exported:', archive)
files.download(archive)

After downloading, follow `BANGLABERT_CHAPTER_TRAINING.md` in the repository. Extract the ZIP into `.runtime/models/pathshongi-banglabert-physics-chapters/`, then run `python -m bangla_rag classify "নিউটনের দ্বিতীয় গতিসূত্র কী?"`.